# Doors Data Analysis and Model Preparation

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Load Dataset

In [ ]:
df = pd.read_csv("doors_final_reclassified.csv")
df.head()

## 3. Initial Inspection

In [ ]:
df.info()
df.describe(include="all")
df.nunique()

## 4. Data Quality Checks

In [ ]:
df.isnull().sum()
df.duplicated().sum()
df[df["Price_EGP"] <= 0]

## 5. Data Cleaning

In [ ]:
df_clean = df.copy()

text_columns = df_clean.select_dtypes(include="object").columns

for col in text_columns:
    df_clean[col] = df_clean[col].str.strip()

df_clean = df_clean.drop_duplicates()
df_clean = df_clean.dropna(subset=["Price_EGP"]).reset_index(drop=True)

df_clean.head()

## 6. Rebuild Quality Levels by Price

In [ ]:
low_threshold = df_clean["Price_EGP"].quantile(1/3)
high_threshold = df_clean["Price_EGP"].quantile(2/3)

df_clean["Quality_Level"] = pd.cut(
    df_clean["Price_EGP"],
    bins=[-np.inf, low_threshold, high_threshold, np.inf],
    labels=["Low", "Medium", "High"],
    include_lowest=True
).astype(str)

df_clean.groupby("Quality_Level")["Price_EGP"].describe()

In [ ]:
quality_price = (
    df_clean.groupby("Quality_Level", as_index=False)["Price_EGP"]
    .mean()
)

quality_price["Quality_Level"] = pd.Categorical(
    quality_price["Quality_Level"],
    categories=["Low", "Medium", "High"],
    ordered=True
)

quality_price = quality_price.sort_values("Quality_Level")

sns.barplot(data=quality_price, x="Quality_Level", y="Price_EGP")
plt.title("Average Door Price by Rebuilt Quality Level")
plt.show()

## 7. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df_clean["Price_EGP"], bins=30, kde=True)
plt.title("Price Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(
    data=df_clean,
    x="Quality_Level",
    y="Price_EGP",
    order=["Low", "Medium", "High"]
)
plt.title("Price Distribution by Quality Level")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=df_clean,
    x="Subcategory",
    y="Price_EGP",
    estimator="mean"
)
plt.xticks(rotation=45)
plt.title("Average Price by Subcategory")
plt.show()

## 8. Outlier Analysis

In [ ]:
Q1 = df_clean["Price_EGP"].quantile(0.25)
Q3 = df_clean["Price_EGP"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_clean[
    (df_clean["Price_EGP"] < lower_bound) |
    (df_clean["Price_EGP"] > upper_bound)
]

outliers.head()

## 9. Door Cost and Quantity Features

In [ ]:
df_model = df_clean.copy()

df_model["Estimated_Quantity"] = np.where(
    df_model["Quantity_Rule"] == "Per_Room",
    df_model["Rule_Value"],
    1
)

df_model["Estimated_Total_Cost"] = (
    df_model["Price_EGP"] * df_model["Estimated_Quantity"]
)

df_model.head()

## 10. Apartment Requirement Logic

In [ ]:
def calculate_doors_requirements(num_rooms, num_balconies=0):
    return {
        "Interior_Doors": num_rooms,
        "Balcony_Doors": num_balconies,
        "Entrance_Doors": 1
    }

In [ ]:
calculate_doors_requirements(
    num_rooms=3,
    num_balconies=2
)

## 11. Cost Estimation by Apartment Requirements

In [ ]:
def estimate_doors_cost(
    data,
    num_rooms,
    num_balconies,
    quality_level=None
):
    requirements = calculate_doors_requirements(
        num_rooms,
        num_balconies
    )

    filtered = data.copy()

    if quality_level is not None:
        filtered = filtered[
            filtered["Quality_Level"] == quality_level
        ]

    interior = filtered[
        filtered["Required_For"] == "Bedroom"
    ]["Price_EGP"].mean() * requirements["Interior_Doors"]

    balcony = filtered[
        filtered["Required_For"] == "Balcony"
    ]["Price_EGP"].mean() * requirements["Balcony_Doors"]

    entrance = filtered[
        filtered["Required_For"] == "Apartment"
    ]["Price_EGP"].mean() * requirements["Entrance_Doors"]

    return interior + balcony + entrance

## 12. Budget-Based Recommendation

In [ ]:
def recommend_doors(
    data,
    budget,
    quality_level,
    num_rooms,
    num_balconies=0
):
    requirements = calculate_doors_requirements(
        num_rooms,
        num_balconies
    )

    filtered = data[
        data["Quality_Level"] == quality_level
    ].copy()

    recommendations = []

    for required_for, quantity in [
        ("Bedroom", requirements["Interior_Doors"]),
        ("Balcony", requirements["Balcony_Doors"]),
        ("Apartment", requirements["Entrance_Doors"])
    ]:
        if quantity > 0:
            options = filtered[
                filtered["Required_For"] == required_for
            ].copy()

            if not options.empty:
                options["Total_Cost"] = (
                    options["Price_EGP"] * quantity
                )

                recommendations.append(
                    options.sort_values("Total_Cost").head(1)
                )

    if recommendations:
        result = pd.concat(recommendations)
        return result[result["Total_Cost"].sum() <= budget] if result["Total_Cost"].sum() <= budget else result

    return pd.DataFrame()

## 13. Multi-File Model Integration Structure

In [ ]:
df_model["Finishing_Category"] = "Doors"

common_columns = [
    "Finishing_Category",
    "Category",
    "Subcategory",
    "Product_Name",
    "Brand",
    "Quality_Level",
    "Price_EGP",
    "Unit",
    "Quantity_Rule",
    "Rule_Value",
    "Required_For",
    "Optional"
]

doors_for_master_model = df_model[
    [col for col in common_columns if col in df_model.columns]
].copy()

doors_for_master_model.head()

## 14. Prepare Data for Future Master Model

In [ ]:
doors_model_data = doors_for_master_model.copy()

doors_model_data = doors_model_data.dropna(
    subset=["Price_EGP"]
).reset_index(drop=True)

doors_model_data.info()

## 15. Door Price Prediction Baseline

In [ ]:
features = [
    "Brand",
    "Product_Name",
    "Door_Category",
    "Material",
    "Door_Type",
    "Application",
    "Size",
    "Quality_Score",
    "Subcategory"
]

features = [
    col for col in features
    if col in df_model.columns
]

X = df_model[features]
y = df_model["Price_EGP"]

categorical_features = X.select_dtypes(
    include="object"
).columns.tolist()

numerical_features = X.select_dtypes(
    exclude="object"
).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=200,
            random_state=42
        ))
    ]
)

## 16. Train and Evaluate Baseline Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

baseline_model.fit(X_train, y_train)

predictions = baseline_model.predict(X_test)

model_results = {
    "MAE": mean_absolute_error(y_test, predictions),
    "RMSE": mean_squared_error(y_test, predictions) ** 0.5,
    "R2": r2_score(y_test, predictions)
}

model_results

## 17. Actual vs Predicted Price

In [ ]:
comparison = pd.DataFrame({
    "Actual_Price": y_test.values,
    "Predicted_Price": predictions
})

comparison.head()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=comparison,
    x="Actual_Price",
    y="Predicted_Price"
)
plt.title("Actual vs Predicted Door Price")
plt.show()

## 18. Final Validation and Export

In [ ]:
doors_model_ready = df_model.copy()

doors_model_ready.to_csv(
    "doors_model_ready_reclassified.csv",
    index=False
)

doors_model_ready.head()

## 19. Final Project Role

هذه البيانات ستكون جزءًا من الـ Master Dataset الخاص بمشروع حساب تكلفة تشطيب شقة كاملة.

الـ Model النهائي سيجمع بيانات:
- Flooring
- Doors
- Paints
- Electrical
- Plumbing
- Lighting
- Ceilings
- وباقي أقسام التشطيب

ثم يعتمد على مدخلات المستخدم مثل:
- Apartment Area
- Number of Rooms
- Budget
- Quality Level
- Preferred Finishing Types

وفي النهاية يحسب:
- الكميات المطلوبة
- تكلفة كل قسم
- أفضل المنتجات المناسبة
- أقل تكلفة ممكنة حسب الاختيارات والميزانية
- التكلفة الإجمالية للتشطيب

ويتم استخدام البيانات المجمعة لاحقًا لتدريب Model واحد باستخدام scikit-learn ثم ربطه بموقع ويب.